
# NeoOLAF native DocRED layer ablation — Skai TV v4

This notebook runs **one complete native NeoOLAF Layer 0–12 pipeline** on one DocRED document.

The v4 patch addresses the v3 diagnosis without changing `src/neoolaf`:

- the whole document is processed in one Layer 1 call;
- Layer 1 emits exact endpoints plus one `SOURCE || PREDICATE || TARGET` relation instance per pair;
- only relation instances use Layer 2 LLM calls; ordinary nodes use deterministic conservative enrichment;
- Layer 2 performs contrastive ontology-property selection with explicit direction rules and synthetic examples;
- Layer 3 injects no empty relation-vocabulary candidates;
- Layer 4 resolves exact structured endpoints deterministically and uses its native parallel LLM task only as fallback;
- coarse ontology/type constraints reject impossible endpoint pairs;
- all prompts, responses, retrievals, decisions, artifacts, logs and errors are saved;
- gold annotations are loaded only after execution.

No direct DocRED extractor, source-entity anchoring, closure rule, or gold-derived fact is used.


In [1]:

from __future__ import annotations

import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from docred_native_ablation_v4 import (
    analyze_run,
    load_layer_states,
    read_json,
    read_jsonl,
    run_native_pipeline,
)

print("PROJECT_ROOT =", PROJECT_ROOT)


c:\Users\henri\Documents\git\post-doc\NeoOLAF\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\henri\Documents\git\post-doc\NeoOLAF\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT = C:\Users\henri\Documents\git\post-doc\NeoOLAF


## 1. Configuration

In [2]:

INPUT_JSONL = NOTEBOOK_DIR / "data/docred_skai_tv_input.jsonl"
GOLD_JSONL = NOTEBOOK_DIR / "data/docred_skai_tv_gold.jsonl"
ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v4.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v4.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/docred_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "skai_tv_structured_contrastive_v4"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Layer 2 relation decisions run concurrently. Layer 4 usually needs no LLM
# call when Layer 1 exact source/target labels resolve cleanly.
WORKERS = 12
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

print("Input:", INPUT_JSONL)
print("Profile:", PROFILE_PATH)
print("Guidance:", GUIDANCE_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))


Input: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\RAGTreeDatasets\data\docred_skai_tv_input.jsonl
Profile: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\RAGTreeDatasets\configs\docred_profile_native_ablation_v4.json
Guidance: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\RAGTreeDatasets\configs\guidance_docred_native_ablation_v4.json
Run dir: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\RAGTreeDatasets\runs\docred_native_layer_ablation\skai_tv_structured_contrastive_v4
Model: openai/gpt-oss-20b
API key available: True


## 2. Preflight: ontology, input guidance, speed configuration

In [3]:

required = [
    INPUT_JSONL, GOLD_JSONL, ONTOLOGY_PATH, ONTOLOGY_ORIGINAL,
    RELATION_CATALOG, RELATION_ALIASES, PROFILE_PATH, GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]

profile = read_json(PROFILE_PATH)
task = input_rows[0]["task_guidance"]
catalog = read_json(RELATION_CATALOG)

print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Ontology properties:", catalog["property_count"])
print("Allowed ontology IDs in input metadata:", len(task["allowed_relation_ids"]))
print("Priority relation IDs:", len(task["priority_relation_ids"]))
print("Relation-instance examples:", len(task["relation_instance_examples"]))
print("Layer 2 workers:", profile["layers"]["layer02_candidate_enrichment"]["max_concurrency"])
print("Layer 4 fallback workers:", profile["layers"]["layer04_candidate_relation_extraction"]["max_concurrency"])
print("Mention-free relation schemas injected:", len(profile["relations"]["allowed"]))

assert len(task["allowed_relation_ids"]) == 96
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_docred_extraction"] is False
assert profile["anti_cheating"]["source_entity_anchoring"] is False


Document: DocRED - e37288ca6012859f - Skai TV
Source characters: 921
Ontology properties: 96
Allowed ontology IDs in input metadata: 96
Priority relation IDs: 21
Relation-instance examples: 4
Layer 2 workers: 12
Layer 4 fallback workers: 8
Mention-free relation schemas injected: 0


## 3. Run the full native pipeline

In [4]:

if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)


[NeoOLAF] Run directory: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\RAGTreeDatasets\runs\docred_native_layer_ablation\skai_tv_structured_contrastive_v4
[NeoOLAF] from_layer=0, to_layer=12, skip_layers=None
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction', 'layer02_candidate_enrichment', 'layer03_candidate_typing_resolution', 'layer04_candidate_relation_extraction', 'layer05_candidate_triple_generation', 'layer06_concept_relation_induction', 'layer07_hierarchisation', 'layer08_axiom_schemata_extraction', 'layer09_general_axiom_extraction', 'layer10_validation_reasoning', 'layer11_inference_completion', 'layer12_serialization']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.00s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extr

[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 0.05s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction
[NeoOLAF][Layer 4] strategy=structured_exact_then_native_parallel_fallback; parallel_workers=8; attempts=1
[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.02s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.00s
[NeoOLAF] Layer 6/12: layer06_concept_relation_induction

[NeoOLAF] Starting layer: layer06_concept_relation_induction
[NeoOLAF][Layer 6] deterministic ontology-aware concept induction for 9 node candidates; no LLM calls.
[NeoOLAF][Layer 6] deterministic ontology-aware relation induction for 8 relation candidates; no LLM calls.
[NeoOLAF] Finished layer: layer06_concept_relation_induction in 0.00s
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontology_aware_axiom_schema_generation
[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.00s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction
[NeoO

[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.00s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion
[NeoOLAF][Layer 11] strategy=ontology_aware_semantic_completion
[NeoOLAF][Layer 11] deterministic completion; max_concurrency=12; no LLM calls.
[NeoOLAF] Finished layer: layer11_inference_completion in 0.00s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\RAGTreeDatasets\runs\docred_native_layer_ablation\skai_tv_structured_contrastive_v4\exports
[NeoOLAF] Finished layer: layer12_serialization in 0.18s
[NeoOLAF] Pipeline finished in 30.93s
[NeoOLAF] Saved checkpoint: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\RAGTreeDatasets\runs\docred_native_layer_ablation\skai_tv_structured_contrastive_v4\checkpoints\after_selected_pipeline.pkl.gz
[NeoOLAF] Total run time: 30.96s
Full native


### Runtime evidence saved

The run directory contains the normal layer states, outputs, metadata, prompts,
checkpoints and exports, plus:

- `run_logs/layer01_relation_instances.json`
- `run_logs/layer02_contrastive_decisions.json`
- `run_logs/layer04_endpoint_assignment.json`
- `run_logs/layer04_constraint_rejections.json`
- `run_logs/llm_calls.jsonl`
- `run_logs/llm_errors.jsonl`
- `run_logs/llm_parse_errors.jsonl`
- `run_logs/ontology_retrieval.jsonl`
- raw responses under `run_logs/responses/`
- `analysis/gold_relation_trace_v4.csv`


## 4. Strict evaluation and cumulative ablation

In [5]:

summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

display(pd.DataFrame(summary["layer_summary"]))

print("Strict final native evaluation")
display(pd.DataFrame([{
    key: summary["strict_evaluation"][key]
    for key in [
        "predicted", "gold", "true_positive", "false_positive",
        "false_negative", "precision", "recall", "f1",
    ]
}]))

print("Cumulative strict evaluation")
display(pd.DataFrame(summary["cumulative_strict_evaluation"]))

print("Corrected v4 first-failure counts")
print(summary["failure_counts_v4"])


,layer_index,layer_name,elapsed_seconds,linguistic_expressions,enriched_expressions,entity_candidates,relation_candidates,attribute_candidates,event_candidates,candidate_relation_assertions,candidate_triples,concept_candidates,ontology_relation_candidates,concept_hierarchy_links,relation_hierarchy_links,axiom_schema_candidates,general_axiom_candidates,completion_candidates,validation_issues,reasoning_inferred_triples
0,0,layer00_preprocessing,0.002,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,layer01_linguistic_expression_extraction,7.219,17,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,layer02_candidate_enrichment,22.278,17,17,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,layer03_candidate_typing_resolution,0.049,17,17,9,8,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4,layer04_candidate_relation_extraction,0.020,17,17,9,8,0,0,8,0,0,0,0,0,0,0,0,0,0
5,5,layer05_candidate_triple_generation,0.002,17,17,9,8,0,0,8,8,0,0,0,0,0,0,0,0,0
6,6,layer06_concept_relation_induction,0.003,17,17,9,8,0,0,8,8,0,7,0,0,0,0,0,0,0
7,7,layer07_hierarchisation,0.001,17,17,9,8,0,0,8,8,0,7,0,7,0,0,0,0,0
8,8,layer08_axiom_schemata_extraction,0.001,17,17,9,8,0,0,8,8,0,7,0,7,21,0,0,0,0
9,9,layer09_general_axiom_extraction,0.001,17,17,9,8,0,0,8,8,0,7,0,7,21,28,0,0,0


Strict final native evaluation


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,7,7,2,5,5,0.285714,0.285714,0.285714


Cumulative strict evaluation


,layer_index,layer_name,mapped_predictions,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,0,layer00_preprocessing,0,0,7,0,0,7,0.000000,0.000000,0.000000
1,1,layer01_linguistic_expression_extraction,0,0,7,0,0,7,0.000000,0.000000,0.000000
2,2,layer02_candidate_enrichment,0,0,7,0,0,7,0.000000,0.000000,0.000000
3,3,layer03_candidate_typing_resolution,0,0,7,0,0,7,0.000000,0.000000,0.000000
4,4,layer04_candidate_relation_extraction,7,7,7,2,5,5,0.285714,0.285714,0.285714
5,5,layer05_candidate_triple_generation,7,7,7,2,5,5,0.285714,0.285714,0.285714
6,6,layer06_concept_relation_induction,7,7,7,2,5,5,0.285714,0.285714,0.285714
7,7,layer07_hierarchisation,7,7,7,2,5,5,0.285714,0.285714,0.285714
8,8,layer08_axiom_schemata_extraction,7,7,7,2,5,5,0.285714,0.285714,0.285714
9,9,layer09_general_axiom_extraction,7,7,7,2,5,5,0.285714,0.285714,0.285714


Corrected v4 first-failure counts
{'layer01_relation_instance_missing': 2, 'layer01_endpoint_missing': 2, 'survived_to_layer05': 2, 'layer02_wrong_or_missing_controlled_relation': 1}


## 5. Layer 1 relation instances and Layer 2 contrastive decisions

In [6]:

layer1_decisions = read_json(RUN_DIR / "run_logs/layer01_relation_instances.json")
layer2_decisions = read_json(RUN_DIR / "run_logs/layer02_contrastive_decisions.json")

print("Layer 1 accepted endpoint/relation expressions")
display(pd.DataFrame(layer1_decisions))

print("Layer 2 canonical property decisions")
display(pd.DataFrame(layer2_decisions))


Layer 1 accepted endpoint/relation expressions


,status,expr_id,text,label,relation_instance,justification
0,accepted,expr_00000,Skai TV,entity_org,None,Named television network.
1,accepted,expr_00001,Piraeus,entity_loc,None,City where Skai TV is based.
2,accepted,expr_00002,Skai Group,entity_org,None,Parent media group.
3,accepted,expr_00003,Greece,entity_loc,None,Country mentioned in the article.
4,accepted,expr_00004,Athens metropolitan area,entity_loc,None,Location of the relaunch.
5,accepted,expr_00005,Nova,entity_org,None,Subscription-based service provider.
6,accepted,expr_00006,Cosmote TV,entity_org,None,Subscription-based service provider.
7,accepted,expr_00007,Digea,entity_org,None,Consortium of private television networks.
8,accepted,expr_00008,1st April 2006,entity_time,None,Date of relaunch.
9,accepted,expr_00009,Skai TV || is based in || Piraeus,relation_instance,"[Skai TV, is based in, Piraeus]",Explicit statement of headquarters location.


Layer 2 canonical property decisions


,expr_id,relation_instance,source,predicate,target,found,selected_relation_id,canonical_relation,decision,ontology_hints
0,expr_00009,Skai TV || is based in || Piraeus,Skai TV,is based in,Piraeus,True,P159,P159 : headquarters location,P159 is preferred because the source is an org...,[controlled_relation:P159 : headquarters locat...
1,expr_00010,Skai TV || is part of || Skai Group,Skai TV,is part of,Skai Group,True,P749,P749 : parent organization,"Skai TV is a subsidiary of Skai Group, so the ...",[controlled_relation:P749 : parent organizatio...
2,expr_00011,Skai TV || was relaunched on || 1st April 2006,Skai TV,was relaunched on,1st April 2006,True,P571,P571 : inception,The source Skai TV is an organization and the ...,"[controlled_relation:P571 : inception, promote..."
3,expr_00012,Skai TV || was relaunched in || Athens metropo...,Skai TV,was relaunched in,Athens metropolitan area,True,P276,P276 : location,The statement describes the location where Ska...,"[controlled_relation:P276 : location, promote_..."
4,expr_00013,Skai TV || is available on || Nova,Skai TV,is available on,Nova,True,P400,P400 : platform,P400 is the most specific property that matche...,"[controlled_relation:P400 : platform, promote_..."
5,expr_00014,Skai TV || is available on || Cosmote TV,Skai TV,is available on,Cosmote TV,True,P400,P400 : platform,P400 is the most specific property that captur...,"[controlled_relation:P400 : platform, promote_..."
6,expr_00015,Skai TV || is a member of || Digea,Skai TV,is a member of,Digea,True,P463,P463 : member of,The source Skai TV is an organization and the ...,"[controlled_relation:P463 : member of, promote..."
7,expr_00016,Skai TV || is a Greek broadcaster || Greece,Skai TV,is a Greek broadcaster,Greece,True,P17,P17 : country,P17 is appropriate because Skai TV is a TV cha...,"[controlled_relation:P17 : country, promote_to..."


## 6. Layer 4 endpoint modes and ontology/type rejections

In [7]:

endpoint_decisions = read_json(RUN_DIR / "run_logs/layer04_endpoint_assignment.json")
constraint_rejections = read_json(RUN_DIR / "run_logs/layer04_constraint_rejections.json")

display(pd.DataFrame(endpoint_decisions))
print("Constraint rejections:", len(constraint_rejections))
if constraint_rejections:
    display(pd.DataFrame(constraint_rejections))

if endpoint_decisions:
    display(pd.DataFrame(endpoint_decisions).groupby(["mode", "llm_call"]).size().reset_index(name="relations"))


,mode,relation_candidate_id,relation_id,source,target,llm_call,type_validation
0,structured_exact,cand_r_00000,P159,Skai TV,Piraeus,False,relation=P159; source_roles=['ORG'] allowed=['...
1,structured_exact,cand_r_00002,P571,Skai TV,1st April 2006,False,relation=P571; source_roles=['ORG'] allowed=['...
2,structured_exact,cand_r_00001,P749,Skai TV,Skai Group,False,relation=P749; source_roles=['ORG'] allowed=['...
3,structured_exact,cand_r_00005,P400,Skai TV,Cosmote TV,False,no_coarse_constraint
4,structured_exact,cand_r_00003,P276,Skai TV,Athens metropolitan area,False,no_coarse_constraint
5,structured_exact,cand_r_00004,P400,Skai TV,Nova,False,no_coarse_constraint
6,structured_exact,cand_r_00006,P463,Skai TV,Digea,False,relation=P463; source_roles=['ORG'] allowed=['...
7,structured_exact,cand_r_00007,P17,Skai TV,Greece,False,relation=P17; source_roles=['ORG'] allowed=['L...


Constraint rejections: 0


,mode,llm_call,relations
0,structured_exact,False,8


## 7. Correct relation-instance trace

In [8]:

trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace_v4.csv")
display(trace_df)
display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)


,gold_relation_id,gold_relation_label,source,target,source_available_layer01,target_available_layer01,relation_instance_layer01,canonical_relation_layer02,relation_candidate_layer03,assertion_layer04,triple_layer05,first_failure
0,P17,P17 : country,Piraeus,Greece,True,True,False,False,False,False,False,layer01_relation_instance_missing
1,P17,P17 : country,Skai Group,Greece,True,True,False,False,False,False,False,layer01_relation_instance_missing
2,P17,P17 : country,Athens,Greece,False,True,False,False,False,False,False,layer01_endpoint_missing
3,P17,P17 : country,Skai TV,Greece,True,True,True,True,True,True,True,survived_to_layer05
4,P159,P159 : headquarters location,Skai TV,Piraeus,True,True,True,True,True,True,True,survived_to_layer05
5,P159,P159 : headquarters location,Skai TV,Athens,True,False,False,False,False,False,False,layer01_endpoint_missing
6,P127,P127 : owned by,Skai TV,Skai Group,True,True,True,False,False,False,False,layer02_wrong_or_missing_controlled_relation


,first_failure,gold_relations
0,layer01_endpoint_missing,2
1,layer01_relation_instance_missing,2
3,survived_to_layer05,2
2,layer02_wrong_or_missing_controlled_relation,1


## 8. Native candidates, assertions, triples and ontology promotion

In [9]:

states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer11 = states.get(11)

if layer3:
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "aliases": c.aliases,
        "controlled_hints": [h for h in c.ontology_hints if str(h).lower().startswith("controlled_relation:")],
    } for c in layer3.relation_candidates or []]))
    assert all(c.mentions for c in layer3.relation_candidates or []), "Mention-free relation candidate detected."

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
        "justification": x.justification,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 triples")
    display(pd.DataFrame([{
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Layer 6 ontology relation candidates:", len(layer6.ontology_relation_candidates or []) if layer6 else None)
print("Layer 11 completion candidates:", len(layer11.completion_candidates or []) if layer11 else None)


,candidate_id,canonical_label,mentions,aliases,controlled_hints
0,cand_r_00000,P159 : headquarters location,[Skai TV || is based in || Piraeus],"[Skai TV || is based in || Piraeus, is based in]",[controlled_relation:P159 : headquarters locat...
1,cand_r_00001,P749 : parent organization,[Skai TV || is part of || Skai Group],"[Skai TV || is part of || Skai Group, is part of]",[controlled_relation:P749 : parent organization]
2,cand_r_00002,P571 : inception,[Skai TV || was relaunched on || 1st April 2006],[Skai TV || was relaunched on || 1st April 200...,[controlled_relation:P571 : inception]
3,cand_r_00003,P276 : location,[Skai TV || was relaunched in || Athens metrop...,[Skai TV || was relaunched in || Athens metrop...,[controlled_relation:P276 : location]
4,cand_r_00004,P400 : platform,[Skai TV || is available on || Nova],"[Skai TV || is available on || Nova, is availa...",[controlled_relation:P400 : platform]
5,cand_r_00005,P400 : platform,[Skai TV || is available on || Cosmote TV],"[Skai TV || is available on || Cosmote TV, is ...",[controlled_relation:P400 : platform]
6,cand_r_00006,P463 : member of,[Skai TV || is a member of || Digea],"[Skai TV || is a member of || Digea, is a memb...",[controlled_relation:P463 : member of]
7,cand_r_00007,P17 : country,[Skai TV || is a Greek broadcaster || Greece],"[Skai TV || is a Greek broadcaster || Greece, ...",[controlled_relation:P17 : country]


Layer 4 assertions


,source,predicate,target,confidence,justification
0,Skai TV,P159 : headquarters location,Piraeus,1.0,Exact Layer 1 source/target labels resolved to...
1,Skai TV,P749 : parent organization,Skai Group,1.0,Exact Layer 1 source/target labels resolved to...
2,Skai TV,P571 : inception,1st April 2006,1.0,Exact Layer 1 source/target labels resolved to...
3,Skai TV,P276 : location,Athens metropolitan area,1.0,Exact Layer 1 source/target labels resolved to...
4,Skai TV,P400 : platform,Nova,1.0,Exact Layer 1 source/target labels resolved to...
5,Skai TV,P400 : platform,Cosmote TV,1.0,Exact Layer 1 source/target labels resolved to...
6,Skai TV,P463 : member of,Digea,1.0,Exact Layer 1 source/target labels resolved to...
7,Skai TV,P17 : country,Greece,1.0,Exact Layer 1 source/target labels resolved to...


Layer 5 triples


,subject,predicate,object,confidence
0,Skai TV,P159 : headquarters location,Piraeus,1.0
1,Skai TV,P749 : parent organization,Skai Group,1.0
2,Skai TV,P571 : inception,1st April 2006,1.0
3,Skai TV,P276 : location,Athens metropolitan area,1.0
4,Skai TV,P400 : platform,Nova,1.0
5,Skai TV,P400 : platform,Cosmote TV,1.0
6,Skai TV,P463 : member of,Digea,1.0
7,Skai TV,P17 : country,Greece,1.0


Layer 6 ontology relation candidates: 7
Layer 11 completion candidates: 0


## 9. Speed, concurrency and errors

In [10]:

def optional_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(path)) if path.is_file() else pd.DataFrame()

calls = optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
retrieval = optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("Backend/API errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)

if not retrieval.empty:
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))

manifest = read_json(RUN_DIR / "run_manifest.json")
print("Wall-clock pipeline seconds:", manifest.get("elapsed_seconds"))


,call_index,layer_tag,model,temperature,message_count,system_chars,user_chars,max_tokens,request_timeout,started_at,status,elapsed_seconds,response_chars,response_path,json_parse_ok,parsed_type,parse_error
0,1,layer01,openai/gpt-oss-20b,0.0,2,1478,7852,4096,120,2026-07-23 02:58:43,ok,7.189,2548,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,True,dict,None
1,8,layer02_relation_only,openai/gpt-oss-20b,0.0,2,901,34273,512,60,2026-07-23 02:58:51,ok,2.649,647,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,True,dict,None
2,4,layer02_relation_only,openai/gpt-oss-20b,0.0,2,901,31496,512,60,2026-07-23 02:58:51,ok,2.751,682,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,True,dict,None
3,7,layer02_relation_only,openai/gpt-oss-20b,0.0,2,901,30532,512,60,2026-07-23 02:58:51,ok,2.939,736,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,True,dict,None
4,6,layer02_relation_only,openai/gpt-oss-20b,0.0,2,901,30526,512,60,2026-07-23 02:58:51,ok,3.822,548,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,True,dict,None
5,3,layer02_relation_only,openai/gpt-oss-20b,0.0,2,901,38331,512,60,2026-07-23 02:58:51,ok,3.976,453,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,True,dict,None
6,2,layer02_relation_only,openai/gpt-oss-20b,0.0,2,901,31679,512,60,2026-07-23 02:58:51,ok,4.629,668,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,True,dict,None
7,9,layer02_relation_only,openai/gpt-oss-20b,0.0,2,901,30535,512,60,2026-07-23 02:58:51,ok,16.443,605,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,True,dict,None
8,5,layer02_relation_only,openai/gpt-oss-20b,0.0,2,901,36088,512,60,2026-07-23 02:58:51,ok,22.131,563,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,True,dict,None


,layer_tag,calls,total_recorded_seconds,maximum_call_seconds,mean_response_chars
0,layer01,1,7.189,7.189,2548.00
1,layer02_relation_only,8,59.340,22.131,612.75


Backend/API errors: 0
JSON parse errors: 0


,layer_name,retrieval_calls
0,layer02_candidate_enrichment,8


Wall-clock pipeline seconds: 30.972



## Success checklist

Before moving to the other four documents, verify that:

1. Layer 1 produces separate temporal and locative relation instances.
2. Layer 2 selects canonical properties with explicit contrastive explanations.
3. Layer 3 contains only document-supported relation candidates.
4. Layer 4 exact endpoint resolution avoids unnecessary LLM calls.
5. No temporal property points to a location.
6. Strict DocRED results and valid-but-unannotated predictions remain separate.
7. Total Layer 2 calls equal the number of relation instances, not all expressions.
